In [4]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'modules')))

In [ ]:
import numpy as np
import torch
from torch import nn
from tqdm import tqdm

import env
import recorder
import metrics

In [ ]:
SG = 24     # Snapshot Group. Hyper Parameter. The number of weather snapshots in a group.

In [ ]:
class ModelZeus(nn.Module):
    def __init__(self, layer_num, d_model, num_heads, drop_rate):
        pass

    def forward(self, x):
        return x


In [ ]:
def validate(model, data_loader, steps):
    model.eval()
    preds_lst, trues_lst, total_loss = [], [], []
    vali_pbar = tqdm(data_loader)
    data_loader.prefetch(steps)     # Fetch one batch asynchronously
    
    for batch_x, batch_y in vali_pbar:  # Wait for the asynchronous .prefetch
        assert tuple(batch_y.shape)[1] == steps * SG    # (B,steps*SG,C,H,W)
        data_loader.prefetch(steps)     # Fetch one batch asynchronously

        batch_x, batch_y = batch_x.to(env.device), batch_y.to(env.device)
        pred_list = []

        for step in range(steps):
            pred_y = model(batch_x, pred_list)   # (B,SG,C,H,W). t is implicit.
            pred_list.append(pred_y)

        pred_y = torch.cat(pred_list, dim=1)   # (B,steps*SG,C,H,W)
        list(map(lambda data, lst: lst.append(data.detach().cpu().numpy()), [
                pred_y, batch_y], [preds_lst, trues_lst]))      # lst = [ (B,steps*SG,C,H,W) ] * n

        loss = env.criterion(pred_y, batch_y)
        vali_pbar.set_description('vali loss: {:.4f}'.format(loss.mean().item()))
        total_loss.append(loss.mean().item())

    total_loss = np.average(total_loss)
    preds = np.concatenate(preds_lst, axis=0)   # (T,B,C,H,W)
    trues = np.concatenate(trues_lst, axis=0)   # (T,B,C,H,W)
    mse, mae, t_mae, ssim, psnr = metrics.metric(preds, trues, data_loader.dataset.mean, data_loader.dataset.std, True)
    #self.t_sample = t_mae/np.sum(t_mae) <- only need for long epoch

    env.print_log('vali mse:{:.4f}, mae:{:.4f}, ssim:{:.4f}, psnr:{:.4f}'.format(mse, mae, ssim, psnr))
    
    return total_loss

In [ ]:
def conclude_epoch(model, tape_recorder, epoch, train_loss, curr_steps, args):
    train_loss = np.average(train_loss)

    with torch.no_grad():
        vali_loss = validate(model, env.vali_loader, curr_steps)
        if epoch % (args.log_step * 100) == 0: env._save(name=str(epoch))
        
    env.print_log("Epoch: {0} | Train Loss: {1:.4f} Vali Loss: {2:.4f}\n".format(
        epoch + 1, train_loss, vali_loss))
    tape_recorder(vali_loss, model, env.path)
    go_next_epoch = True
    
    return go_next_epoch

In [ ]:
def train(model, data_loader, args):
    config = args.__dict__
    tape_recorder = recorder.Recorder(verbose=True)

    for epoch in range(config['epochs']):
        model.train()
        train_loss = []
        train_pbar = tqdm(env.train_loader)

        new_steps = env.get_next_steps()
        data_loader.prefetch(new_steps)        # Fetch one batch asynchronously

        for batch_x, batch_y in train_pbar:         # Wait for the asynchronous .prefetch
            # batch_x: (B,T1*SG,C,H,W), batch_y: (B,T2*SG,C,H,W), T2 = new_steps
            curr_steps = new_steps
            assert tuple(batch_y.shape)[1] == curr_steps * SG
            new_steps = env.get_next_steps()
            data_loader.prefetch(new_steps)    # Fetch one batch asynchronously

            batch_x, batch_y = batch_x.to(env.device), batch_y.to(env.device)
            pred_list = []

            for step in range(curr_steps):
                pred_y = model(batch_x, pred_list)   # (B,SG,C,H,W). t is implicit.
                target = batch_y[:, step*SG : (step+1)*SG, :, :, :]
                pred_list.append(pred_y)   # instead of pred_y
                loss = env.criterion(pred_y, target)
                train_loss.append(loss.item())
                train_pbar.set_description('train loss: {:.4f}'.format(loss.item()))

                env.optimizer.zero_grad()
                loss.backward()             
                env.optimizer.step()

            env.scheduler.step()

            if env.step % env.update_ema_every == 0: env.step_ema()
            env.step += 1

        go_next_epoch = conclude_epoch(model, tape_recorder, epoch, train_loss, curr_steps, args)
        if not go_next_epoch: break

    best_model_path = env.path + '/' + 'checkpoint.pth'
    model.load_state_dict(torch.load(best_model_path))
    return model

In [ ]:
def test(model, data_loader):
    model.eval()
    preds_lst, trues_lst, total_loss = [], [], []
    vali_pbar = tqdm(data_loader)

    new_steps = env.get_next_steps()
    data_loader.prefetch(new_steps)        # Fetch one batch asynchronously
    
    for batch_x, batch_y in vali_pbar:         # Wait for the asynchronous .prefetch
        curr_steps = new_steps
        assert tuple(batch_y.shape)[1] == curr_steps * SG   # (B,T2,C,H,W), T2 = curr_steps 
        new_steps = env.get_next_steps()
        data_loader.prefetch(new_steps)    # Fetch one batch asynchronously

        pred_list = []
        for step in range(curr_steps):
            pred_y = model(batch_x.to(env.device), pred_list)    # (B,SG,C,H,W). t is implicit.
            pred_list.append(pred_y)

        pred_y = torch.cat(pred_list, dim=1).unsqueeze(2)
        list(map(lambda data, lst: lst.append(data.detach().cpu().numpy()), [
                pred_y, batch_y], [preds_lst, trues_lst]))  # lst = [ (B,steps*SG,C,H,W) ] * n

        loss = env.criterion(pred_y, batch_y)
        vali_pbar.set_description('test loss: {:.4f}'.format(loss.mean().item()))
        total_loss.append(loss.mean().item())

    total_loss = np.average(total_loss)
    preds = np.concatenate(preds_lst, axis=0)
    trues = np.concatenate(trues_lst, axis=0)
    mse, mae, t_mae, ssim, psnr = metrics.metric(preds, trues, data_loader.dataset.mean, data_loader.dataset.std, True)
    #self.t_sample = t_mae/np.sum(t_mae) <- only need for long epoch

    env.print_log('test mse:{:.4f}, mae:{:.4f}, ssim:{:.4f}, psnr:{:.4f}'.format(mse, mae, ssim, psnr))
    
    return total_loss

In [ ]:
def inference(model, lat, lon, start_hour, hours):
    # sketch only
    x, t_date = get_latest_truth_data()     # x: (T1,C,H,W)
    steps = get_steps(t_date, start_hour, hours)

    batch_x = x.unsqueeze(0)    # (B=1,T1,C,H,W)

    pred_list = []
    for step in range(steps):
        pred_y = model(batch_x.to(env.device), pred_list)    # (B,SG,C,H,W). t is implicit.
        pred_list.append(pred_y)
    
    response = get_response(pred_list, t_date, lat, lon, start_hour, hours)
    response = format_zeuse(response)

    return response

    


In [ ]:
args = {}
data_loader, _, _ = get_data_loader(SG)
theModel = ModelZeus().to(env.device)

train(theModel, data_loader, args)